# Al Meezan Qatar Legal Data - Machine Learning Analysis

This notebook demonstrates how to use the scraped legal data for machine learning applications.

## Table of Contents
1. Load and Explore Data
2. Data Preprocessing
3. Feature Engineering
4. Text Classification
5. Topic Modeling
6. Time Series Analysis
7. Visualization

## 1. Setup and Data Loading

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
# Load the scraped data
# Option 1: Load raw data
df_raw = pd.read_csv('qatar_legal_data.csv')

# Option 2: Load ML-ready data (recommended)
df = pd.read_csv('qatar_legal_data_ml_ready.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
df.head()

## 2. Data Exploration

In [ ]:
# Basic statistics
print("=== Data Summary ===")
print(f"Total records: {len(df)}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nData types:\n{df.dtypes}")

In [ ]:
# Distribution by category
category_counts = df['category'].value_counts()
print("\nRecords by Category:")
print(category_counts)

# Visualize
plt.figure(figsize=(10, 6))
category_counts.plot(kind='bar', color=['#1f77b4', '#ff7f0e', '#2ca02c'])
plt.title('Distribution of Legal Documents by Category', fontsize=14, fontweight='bold')
plt.xlabel('Category')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Year distribution
if 'year' in df.columns:
    plt.figure(figsize=(14, 6))
    df['year'].value_counts().sort_index().plot(kind='line', marker='o')
    plt.title('Legal Documents Published Over Time', fontsize=14, fontweight='bold')
    plt.xlabel('Year')
    plt.ylabel('Number of Documents')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print(f"\nYear Range: {df['year'].min()} - {df['year'].max()}")
    print(f"Mean Year: {df['year'].mean():.0f}")

In [ ]:
# Text length statistics
if 'title_length' in df.columns:
    print("\n=== Text Statistics ===")
    print(f"Average title length: {df['title_length'].mean():.1f} characters")
    print(f"Average word count: {df['title_word_count'].mean():.1f} words")
    
    # Visualize distribution
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    axes[0].hist(df['title_length'], bins=30, color='skyblue', edgecolor='black')
    axes[0].set_title('Distribution of Title Length')
    axes[0].set_xlabel('Characters')
    axes[0].set_ylabel('Frequency')
    
    axes[1].hist(df['title_word_count'], bins=20, color='lightcoral', edgecolor='black')
    axes[1].set_title('Distribution of Word Count')
    axes[1].set_xlabel('Words')
    axes[1].set_ylabel('Frequency')
    
    plt.tight_layout()
    plt.show()

## 3. Text Classification Example

Classify legal documents by category using their titles.

In [ ]:
# Prepare data for classification
# Filter out rows with missing titles or categories
df_clean = df.dropna(subset=['title', 'category'])

print(f"Clean dataset size: {len(df_clean)} records")

In [ ]:
# Feature extraction using TF-IDF
vectorizer = TfidfVectorizer(max_features=500, min_df=2, max_df=0.8)
X_text = vectorizer.fit_transform(df_clean['title'])

print(f"TF-IDF matrix shape: {X_text.shape}")
print(f"Top 10 features: {vectorizer.get_feature_names_out()[:10]}")

In [ ]:
# Combine text features with numerical features
numerical_features = ['year', 'title_length', 'title_word_count']
available_features = [f for f in numerical_features if f in df_clean.columns]

if available_features:
    X_numeric = df_clean[available_features].fillna(0)
    X = np.hstack([X_text.toarray(), X_numeric.values])
else:
    X = X_text.toarray()

# Target variable
y = df_clean['category_encoded'] if 'category_encoded' in df_clean.columns else pd.Categorical(df_clean['category']).codes

print(f"Feature matrix shape: {X.shape}")
print(f"Target distribution: {pd.Series(y).value_counts().to_dict()}")

In [ ]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")

In [ ]:
# Train Random Forest Classifier
clf = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

print("Training classifier...")
clf.fit(X_train, y_train)

# Make predictions
y_pred = clf.predict(X_test)

# Evaluate
accuracy = accuracy_score(y_test, y_pred)
print(f"\n=== Model Performance ===")
print(f"Accuracy: {accuracy:.2%}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=df_clean['category'].unique()))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=df_clean['category'].unique(),
            yticklabels=df_clean['category'].unique())
plt.title('Confusion Matrix', fontsize=14, fontweight='bold')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance (for numerical features)
if available_features:
    # Get feature importance for the numerical features
    n_text_features = X_text.shape[1]
    feature_importance = clf.feature_importances_[n_text_features:]
    
    importance_df = pd.DataFrame({
        'feature': available_features,
        'importance': feature_importance
    }).sort_values('importance', ascending=False)
    
    print("\nNumerical Feature Importance:")
    print(importance_df)
    
    plt.figure(figsize=(10, 6))
    plt.barh(importance_df['feature'], importance_df['importance'])
    plt.xlabel('Importance')
    plt.title('Feature Importance (Numerical Features)', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 4. Time Series Analysis

In [ ]:
# Analyze trends over time
if 'year' in df.columns and 'category' in df.columns:
    year_category = df.groupby(['year', 'category']).size().unstack(fill_value=0)
    
    plt.figure(figsize=(14, 6))
    year_category.plot(kind='area', stacked=True, alpha=0.7)
    plt.title('Legal Documents Published by Category Over Time', fontsize=14, fontweight='bold')
    plt.xlabel('Year')
    plt.ylabel('Number of Documents')
    plt.legend(title='Category', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

## 5. Advanced Analysis - Decade Trends

In [ ]:
# Analyze by decade
if 'decade' in df.columns:
    decade_summary = df.groupby('decade').agg({
        'title': 'count',
        'category': lambda x: x.mode()[0] if len(x.mode()) > 0 else 'N/A'
    }).rename(columns={'title': 'count', 'category': 'most_common_category'})
    
    print("\n=== Legislation by Decade ===")
    print(decade_summary)
    
    plt.figure(figsize=(12, 6))
    decade_summary['count'].plot(kind='bar', color='teal')
    plt.title('Number of Legal Documents by Decade', fontsize=14, fontweight='bold')
    plt.xlabel('Decade')
    plt.ylabel('Count')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 6. Export Results

In [ ]:
# Save analysis results
results = {
    'total_documents': len(df),
    'categories': df['category'].value_counts().to_dict() if 'category' in df.columns else {},
    'year_range': f"{df['year'].min()} - {df['year'].max()}" if 'year' in df.columns else 'N/A',
    'classification_accuracy': f"{accuracy:.2%}" if 'accuracy' in locals() else 'N/A'
}

import json
with open('analysis_results.json', 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print("Analysis results saved to 'analysis_results.json'")
print(json.dumps(results, indent=2, ensure_ascii=False))

## 7. Next Steps

Potential ML applications:

1. **Document Clustering**: Group similar legal documents using K-Means or DBSCAN
2. **Topic Modeling**: Extract topics using LDA or NMF
3. **Named Entity Recognition**: Extract legal entities, dates, and references
4. **Document Summarization**: Generate summaries of legal texts
5. **Similarity Search**: Find similar laws based on content
6. **Trend Prediction**: Forecast future legislation patterns
7. **Recommendation System**: Recommend related legal documents

For Arabic NLP:
- Use libraries like `CAMeL Tools`, `Farasa`, or `AraBERT`
- Apply Arabic-specific preprocessing (stemming, stopword removal)
- Fine-tune transformer models for Arabic legal text